In [2]:
import sys
!{sys.executable} -m pip install usaddress

  Using cached usaddress-0.5.16-py3-none-any.whl.metadata (6.7 kB)
  Using cached probableparsing-0.0.1-py2.py3-none-any.whl.metadata (908 bytes)
Using cached usaddress-0.5.16-py3-none-any.whl (70 kB)
Using cached probableparsing-0.0.1-py2.py3-none-any.whl (3.1 kB)

   ---------------------------------------- 3/3 [usaddress]



In [1]:
import pandas as pd
import usaddress

def parse_messy_record(raw_text):
    """
    Parses a single messy string into distinct address components.
    Gracefully handles missing data and parser confusion.
    """
    # Initialize our clean target dictionary
    parsed_data = {
        'Name': '',
        'Address': '',
        'City': '',
        'State': '',
        'Zip': ''
    }
    
    # Catch empty or null rows immediately
    if pd.isna(raw_text) or not str(raw_text).strip():
        return pd.Series(parsed_data)
        
    try:
        # usaddress.tag attempts to group the parsed components intelligently
        tagged_dict, address_type = usaddress.tag(str(raw_text))
        
        # 1. Extract the easy components
        parsed_data['Name'] = tagged_dict.get('Recipient', '')
        parsed_data['City'] = tagged_dict.get('PlaceName', '')
        parsed_data['State'] = tagged_dict.get('StateName', '')
        parsed_data['Zip'] = tagged_dict.get('ZipCode', '')
        
        # 2. Reconstruct the street address
        # The library breaks addresses down into highly granular pieces (e.g., StreetNamePreDirectional)
        # We need to glue the relevant bits back together for a standard Address Line 1.
        address_parts = [
            tagged_dict.get('AddressNumber', ''),
            tagged_dict.get('StreetNamePreDirectional', ''),
            tagged_dict.get('StreetName', ''),
            tagged_dict.get('StreetNamePostType', ''),
            tagged_dict.get('OccupancyType', ''),      # e.g., 'Apt', 'Suite'
            tagged_dict.get('OccupancyIdentifier', '') # e.g., 'B', '100'
        ]
        
        # Filter out blanks and stitch together with spaces
        parsed_data['Address'] = ' '.join(part for part in address_parts if part).strip()
        
    except usaddress.RepeatedLabelError:
        # FAILSAFE: If the string is so mangled that the ML model gets confused 
        # (e.g., it thinks there are two cities), it throws this error.
        # We flag it for manual review rather than crashing the script.
        parsed_data['Name'] = "FLAG: MANUAL REVIEW"
        parsed_data['Address'] = raw_text 
        
    return pd.Series(parsed_data)

# ==========================================
# Execution Pipeline
# ==========================================

def clean_file(input_path, output_path, column_name):
    print(f"Loading data from {input_path}...")
    
    # Read the file (handles both CSV and Excel)
    if input_path.endswith('.csv'):
        df = pd.read_csv(input_path)
    else:
        df = pd.read_excel(input_path)
        
    if column_name not in df.columns:
        raise ValueError(f"Column '{column_name}' not found in the dataset.")

    print(f"Parsing column '{column_name}'. This may take a moment for large files...")
    
    # Apply the parser to the messy column
    # This creates a new DataFrame with our 5 clean columns
    parsed_columns = df[column_name].apply(parse_messy_record)
    
    # Merge the clean data back with the original dataset (optional, but good for auditing)
    result_df = pd.concat([df, parsed_columns], axis=1)
    
    # Save the output
    if output_path.endswith('.csv'):
        result_df.to_csv(output_path, index=False)
    else:
        result_df.to_excel(output_path, index=False)
        
    print(f"Processing complete! Saved to {output_path}")

# --- Example Usage ---
# clean_file('messy_client_data.xlsx', 'cleaned_data_v1.xlsx', 'Raw_Contact_Info')

In [2]:
# 1. Update this to the exact name of your test file (include .csv or .xlsx)
# If the file isn't in the same folder as this notebook, you'll need the full file path.
input_file = r"C:\Users\8DATA2\Desktop\Working Folder\Data Cleanup 1 Column\Valley People Celebration Invites - without duplicates.csv"

# 2. What do you want to name the finished file?
output_file = r"C:\Users\8DATA2\Desktop\Working Folder\Data Cleanup 1 Column\Valley People Celebration Invites - without FIXED.csv"

# 3. CRITICAL: Type the exact column header of the messy data here (case-sensitive)
target_column = "Mailing Address"

# Run the pipeline
clean_file(input_file, output_file, target_column)

Loading data from C:\Users\8DATA2\Desktop\Working Folder\Data Cleanup 1 Column\Valley People Celebration Invites - without duplicates.csv...
Parsing column 'Mailing Address'. This may take a moment for large files...
Processing complete! Saved to C:\Users\8DATA2\Desktop\Working Folder\Data Cleanup 1 Column\Valley People Celebration Invites - without FIXED.csv
